# PS S6E6 — LightGBM v2-fix

**v2 OOF:** 0.96514 | **v2 LB:** 0.96576  
**Problem with v2:** Optuna found `lr=0.0136` → all folds hit `best_iter=2000`, model never converged.  
**Fix:**
- Optuna LR floor raised: `0.01 → 0.03` (avoids pathologically slow convergence)
- Optuna search trees: `500 → 800` (ensures convergence within the search itself)
- Final trees: `2000 → 3000` (ceiling high enough for any LR found)

Same features as v2. Saves `test_proba` and `oof_proba` as CSV for v5 ensemble.

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import optuna

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
CFG = dict(
    n_folds         = 5,
    seed            = 42,
    optuna_trials   = 50,
    optuna_folds    = 3,
    optuna_sample   = 0.3,
    optuna_trees    = 800,   # was 500 — must converge within search
    n_estimators    = 3000,  # was 2000 — ceiling for final model
    early_stop      = 50,
)

V2_OOF = 0.96514
V2_LB  = 0.96576

## 2. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/ekowannanindome/stellar-dataset/train.csv', index_col='id')
test  = pd.read_csv('/kaggle/input/datasets/ekowannanindome/stellar-dataset/test.csv',  index_col='id')
print(f'Train: {train.shape}  |  Test: {test.shape}')

## 3. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']
    df['u_z']          = df['u'] - df['z']
    df['log_redshift'] = np.log1p(df['redshift'])
    df['redshift_sq']  = df['redshift'] ** 2
    df['rs_x_gr']      = df['redshift'] * df['g_r']
    df['rs_x_ug']      = df['redshift'] * df['u_g']
    df['rs_x_gi']      = df['redshift'] * df['g_i']
    return df

train = engineer_features(train)
test  = engineer_features(test)

CAT_COLS  = ['spectral_type', 'galaxy_population']
NUM_COLS  = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
V1_COLS   = ['u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'g_z']
V2_COLS   = ['u_z', 'log_redshift', 'redshift_sq', 'rs_x_gr', 'rs_x_ug', 'rs_x_gi']
FEAT_COLS = NUM_COLS + V1_COLS + V2_COLS + CAT_COLS
TARGET    = 'class'

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[CAT_COLS] = oe.fit_transform(train[CAT_COLS])
test[CAT_COLS]  = oe.transform(test[CAT_COLS])

le = LabelEncoder()
y  = le.fit_transform(train[TARGET])
print('Classes:', dict(zip(le.classes_, le.transform(le.classes_))))

X      = train[FEAT_COLS]
X_test = test[FEAT_COLS]
print(f'Feature matrix: {X.shape}')

## 4. Optuna Search (fixed LR floor)

In [ ]:
_, X_opt, _, y_opt = train_test_split(
    X, y, test_size=CFG['optuna_sample'], stratify=y, random_state=CFG['seed']
)
print(f'Optuna search set: {X_opt.shape}')

def objective(trial):
    params = dict(
        n_estimators      = CFG['optuna_trees'],
        learning_rate     = trial.suggest_float('learning_rate', 0.03, 0.15, log=True),  # floor raised
        num_leaves        = trial.suggest_int('num_leaves', 63, 511),
        min_child_samples = trial.suggest_int('min_child_samples', 20, 200),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        class_weight      = 'balanced',
        n_jobs=-1, random_state=CFG['seed'], verbose=-1,
    )
    skf = StratifiedKFold(n_splits=CFG['optuna_folds'], shuffle=True, random_state=CFG['seed'])
    scores = []
    for tr_idx, val_idx in skf.split(X_opt, y_opt):
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_opt.iloc[tr_idx], y_opt[tr_idx],
            eval_set=[(X_opt.iloc[val_idx], y_opt[val_idx])],
            callbacks=[lgb.early_stopping(30, verbose=False)],
            categorical_feature=CAT_COLS,
        )
        preds = le.inverse_transform(m.predict(X_opt.iloc[val_idx]))
        truth = le.inverse_transform(y_opt[val_idx])
        scores.append(balanced_accuracy_score(truth, preds))
    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=CFG['seed']))
study.optimize(objective, n_trials=CFG['optuna_trials'], show_progress_bar=True)

print(f'\nBest trial score: {study.best_value:.5f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## 5. Final 5-Fold CV

In [ ]:
BEST_PARAMS = dict(
    **study.best_params,
    n_estimators=CFG['n_estimators'],
    class_weight='balanced',
    n_jobs=-1, random_state=CFG['seed'], verbose=-1,
)

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
oof_proba  = np.zeros((len(X), len(le.classes_)))
test_proba = np.zeros((len(X_test), len(le.classes_)))
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = lgb.LGBMClassifier(**BEST_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(CFG['early_stop'], verbose=False),
            lgb.log_evaluation(200),
        ],
        categorical_feature=CAT_COLS,
    )
    val_proba = model.predict_proba(X_val)
    val_preds = le.inverse_transform(val_proba.argmax(axis=1))
    score = balanced_accuracy_score(le.inverse_transform(y_val), val_preds)
    fold_scores.append(score)
    oof_proba[val_idx] = val_proba
    test_proba += model.predict_proba(X_test) / CFG['n_folds']

    print(f'  Fold {fold+1} | best_iter={model.best_iteration_} | balanced_acc={score:.5f}')

print(f'\nCV mean: {np.mean(fold_scores):.5f} | std: {np.std(fold_scores):.5f}')

## 6. OOF Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

oof_labels  = le.inverse_transform(oof_proba.argmax(axis=1))
true_labels = le.inverse_transform(y)
oof_score   = balanced_accuracy_score(true_labels, oof_labels)

delta = oof_score - V2_OOF
print(f'OOF: {oof_score:.5f}  |  v2 OOF: {V2_OOF:.5f}  |  delta: {"+" if delta>=0 else ""}{delta:.5f}')

classes_ordered = ['GALAXY', 'QSO', 'STAR']
print()
print(classification_report(true_labels, oof_labels, target_names=classes_ordered))

ConfusionMatrixDisplay(
    confusion_matrix(true_labels, oof_labels, labels=classes_ordered),
    display_labels=classes_ordered
).plot(cmap='Blues')
plt.title(f'v2-fix OOF — {oof_score:.4f}')
plt.tight_layout()

## 7. Feature Importance

In [ ]:
imp = pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
imp = imp.sort_values('importance', ascending=False)

imp.plot.barh(x='feature', y='importance', figsize=(10, 10), legend=False)
plt.gca().invert_yaxis()
plt.title('v2-fix Feature Importance')
plt.tight_layout()

print(imp.to_string(index=False))

## 8. Save Probabilities + Submission

In [ ]:
# Save for ensemble (v5)
pd.DataFrame(oof_proba,  columns=le.classes_).to_csv('oof_proba_lgbm.csv',  index=False)
pd.DataFrame(test_proba, columns=le.classes_).to_csv('test_proba_lgbm.csv', index=False)

# Submission
test_pred_labels = le.inverse_transform(test_proba.argmax(axis=1))
submission = pd.DataFrame({'id': test.index, 'class': test_pred_labels})
submission.to_csv('submission_lgbm_v2fix.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(submission['class'].value_counts())

In [ ]:
from IPython.display import FileLink, display
display(FileLink('submission_lgbm_v2fix.csv'))

## 9. Results

| Model | OOF | LB | Delta OOF | Notes |
|---|---|---|---|---|
| LightGBM v2 | 0.96514 | 0.96576 | — | lr=0.0136, hit iter ceiling |
| LightGBM v2-fix | ... | ... | ... | LR floor 0.03, 3000 trees |

**Did early stopping fire this time? (best_iter < 3000):**  
**STAR precision vs v2 (was 0.88):**  
**Best LR found by Optuna:**